# CM3070 Final Project
# End-to-End Pipeline: Model 1 -> Model 2 -> Model 3

This notebook chains all three stages of the Contract Analysis AI system on a single contract:
1. **Model 1** — PDF extraction + clause chunking (`pdfplumber`, with a Tesseract OCR fallback for scanned pages)
2. **Model 2** — clause classification (zero-shot DeBERTa NLI; a separately fine-tuned LEGAL-BERT alternative exists and is validated in `Model2_LegalBERT_FineTuning.ipynb`, but not used here)
3. **Model 3** — LLM-based, clause-specific explanation + risk assessment (Groq API)

The point of this notebook isn't to improve any single stage since each was already validated on its own. It's to catch **integration seams**: places where one stage's real output doesn't quite match what the next stage assumed, which only shows up when you actually run them together.

# Cell 1 - Install All Dependencies

In [ ]:
!apt-get install -y tesseract-ocr poppler-utils -qq
!pip install pdfplumber reportlab transformers torch groq pandas tqdm pytesseract pdf2image -q

# Cell 2 - Import Libraries

In [ ]:
import re
import json
import time
import getpass
import warnings
import pandas as pd
import pytesseract
from pytesseract import Output
from pdf2image import convert_from_path
from google.colab import files
warnings.filterwarnings('ignore')

# ============================================================
# MODEL 1: DOCUMENT PROCESSING
# ============================================================

# Cell 3 - Get a Contract PDF (upload your own or generate a synthetic sample)

Upload a real contract for a genuine end-to-end test. Skipping falls back to a synthetic sample -- useful for confirming the pipeline runs but a synthetic contract does not show whether Model 1's extraction actually holds up on messy real-world PDF formatting.

In [ ]:
def get_contract_pdf():
    print("Upload a contract PDF, or skip to use a synthetic sample instead.")
    try:
        uploaded = files.upload()
        if uploaded:
            pdf_path = list(uploaded.keys())[0]
            print(f"\nUsing uploaded file: {pdf_path}")
            return pdf_path
    except Exception as e:
        print(f"No file uploaded ({e}). Falling back to sample contract.")
    print("No upload detected — generating a synthetic sample contract instead.")
    return generate_sample_contract()


def generate_sample_contract(path="sample_employment_contract.pdf"):
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.units import mm
    from reportlab.lib.styles import getSampleStyleSheet
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
    from reportlab.lib.enums import TA_JUSTIFY

    doc = SimpleDocTemplate(path, pagesize=A4, topMargin=20*mm, bottomMargin=20*mm,
                             leftMargin=20*mm, rightMargin=20*mm)
    styles = getSampleStyleSheet()
    styles['Normal'].alignment = TA_JUSTIFY
    story = [Paragraph("EMPLOYMENT AGREEMENT", styles['Title']), Spacer(1, 12),
             Paragraph("This Employment Agreement is made between Acme Technologies "
                       "Pte Ltd (\"the Company\") and Jane Tan (\"the Employee\").",
                       styles['Normal']), Spacer(1, 12)]
    clauses = [
        ("1. Position and Duties", "The Employee shall be employed as a Senior Software Engineer and shall report to the Head of Engineering."),
        ("2. Compensation and Benefits", "The Employee shall receive a monthly salary of SGD 6,500, payable on the last working day of each month, together with an annual performance bonus at the discretion of the Company."),
        ("3. Probationary Period", "The Employee's first three months of employment shall be a probationary period, during which either party may terminate this contract with one week's notice."),
        ("4. Termination", "Either party may terminate this Agreement by giving one month's written notice. The Company reserves the right to terminate employment immediately for gross misconduct."),
        ("5. Confidentiality", "The Employee agrees to keep confidential all trade secrets, business strategies, client information, and proprietary data obtained during employment."),
        ("6. Intellectual Property", "All inventions, developments, software, and intellectual property created by the Employee in the course of employment shall be the exclusive property of the Company."),
        ("7. Non-Compete", "The Employee shall not, during the term of employment and for a period of 12 months thereafter, directly or indirectly engage in any business that competes with the Company within Singapore."),
        ("8. Governing Law", "This Agreement shall be governed by and construed in accordance with the laws of Singapore."),
    ]
    for heading, body in clauses:
        story.append(Paragraph(heading, styles['Heading2']))
        story.append(Spacer(1, 6))
        story.append(Paragraph(body, styles['Normal']))
        story.append(Spacer(1, 10))
    doc.build(story)
    print(f"Sample contract generated: {path}")
    return path


pdf_path = get_contract_pdf()

# Cell 4 - Model 1: Extraction, Cleaning & Chunking

v4 logic, validated against a real MOM contract template and two genuine SEC EDGAR employment contracts. Includes running header/footer stripping, closing/signature-block detection, placeholder filtering, fragment-vs-real-clause disambiguation by capitalisation instead of just length alone and a Tesseract OCR fallback (this pipeline's third pre-trained model) for pages with no extractable text layer.

In [ ]:
import pdfplumber
from collections import defaultdict

def needs_ocr(pdf_path, min_chars_per_page=20):
    """Detects pages with no meaningful extractable text layer."""
    pages_needing_ocr = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages, start=1):
            text = page.extract_text() or ""
            if len(text.strip()) < min_chars_per_page:
                pages_needing_ocr.append(i)
    return pages_needing_ocr


def ocr_page_to_lines(image, page_num, dpi=200):
    """
    Runs Tesseract's LSTM OCR engine (a genuine pre-trained neural OCR
    model -- this pipeline's supplementary model alongside Model 2's
    classifier and Model 3's LLM) on a page image, grouping word-level
    output into lines with positions rescaled into points so gaps are
    directly comparable to pdfplumber's native coordinate system.
    """
    data = pytesseract.image_to_data(image, output_type=Output.DICT)
    lines = {}
    for i in range(len(data['text'])):
        word = data['text'][i].strip()
        if not word:
            continue
        key = (data['block_num'][i], data['par_num'][i], data['line_num'][i])
        top, height = data['top'][i], data['height'][i]
        if key not in lines:
            lines[key] = {'words': [], 'top': top, 'bottom': top + height}
        lines[key]['words'].append(word)
        lines[key]['bottom'] = max(lines[key]['bottom'], top + height)

    scale = 72.0 / dpi
    ordered = sorted(lines.values(), key=lambda l: l['top'])
    out, prev_bottom = [], None
    for l in ordered:
        top_pt, bottom_pt = l['top'] * scale, l['bottom'] * scale
        gap = (top_pt - prev_bottom) if prev_bottom is not None else None
        out.append({'page': page_num, 'text': ' '.join(l['words']), 'gap_before': gap})
        prev_bottom = bottom_pt
    return out


def extract_pages(pdf_path, ocr_dpi=200):
    all_lines = []
    ocr_needed_pages = set(needs_ocr(pdf_path))

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            if page_num in ocr_needed_pages:
                continue
            try:
                lines = page.extract_text_lines()
            except Exception:
                lines = []
            prev_bottom = None
            for l in lines:
                text = l.get('text', '').strip()
                if not text:
                    continue
                gap = (l['top'] - prev_bottom) if prev_bottom is not None else None
                all_lines.append({'page': page_num, 'text': text, 'gap_before': gap})
                prev_bottom = l['bottom']

    if ocr_needed_pages:
        print(f"Pages with no extractable text layer -- running OCR: {sorted(ocr_needed_pages)}")
        images = convert_from_path(pdf_path, dpi=ocr_dpi)
        for page_num in sorted(ocr_needed_pages):
            all_lines.extend(ocr_page_to_lines(images[page_num - 1], page_num, dpi=ocr_dpi))
        all_lines.sort(key=lambda l: l['page'])

    return all_lines


_STAMP_PATTERNS = [
    re.compile(r'Updated\s+on\s+\d{1,2}[/\-]\d{1,2}[/\-]\d{2,4}', re.I),
    re.compile(r'^[A-Z]-\d+$'),
    re.compile(r'^\d{1,2}[/\-]\d{1,2}[/\-]\d{2,4}\s+\d{1,2}:\d{2}(:\d{2})?$'),
]

def strip_boilerplate(all_lines, min_page_repeats=2):
    def norm(t):
        return re.sub(r'\s+', ' ', t.strip().lower())
    page_sets = defaultdict(set)
    for rec in all_lines:
        page_sets[norm(rec['text'])].add(rec['page'])
    repeated = {t for t, pages in page_sets.items() if len(pages) >= min_page_repeats}
    out = []
    for rec in all_lines:
        text = rec['text']
        if norm(text) in repeated:
            continue
        if any(p.search(text) for p in _STAMP_PATTERNS):
            continue
        out.append(rec)
    return out


def clean_line(text):
    text = re.sub(r'^\s*Page\s+\d+(\s+of\s+\d+)?\s*$', '', text)
    text = re.sub(r'^\s*\d{1,3}\s*$', '', text)
    return text.strip()


HEADING_PATTERNS = [
    re.compile(r'^\s*(\d{1,2})\.?\s+([A-Z][A-Za-z0-9,/&\'\-\s]{2,80})\s*$'),
    re.compile(r'^\s*(ARTICLE|Article)\s+([IVXLCDM]+|\d+)\b[\s:.\-]*(.*)$'),
    re.compile(r'^\s*(SECTION|Section)\s+(\d+)\b[\s:.\-]*(.*)$'),
    re.compile(r'^\s*[A-Z][A-Z\s]{3,60}\s*$'),
]
_NON_HEADING_STARTERS = ('PLEASE ', 'NOTE ', 'NOTE:', 'WARNING', 'IMPORTANT',
                          'CAUTION', 'ATTENTION', 'DISCLAIMER')
_CLOSING_MARKERS = re.compile(r'\b(IN WITNESS WHEREOF|SIGNED AT|SIGNATURE[S]?\s*:?\s*$)\b', re.I)


def is_heading(line):
    line = line.strip()
    if not line or len(line) > 140:
        return False
    if _CLOSING_MARKERS.search(line):
        return True
    if len(line) > 100:
        return False
    if line.upper().startswith(_NON_HEADING_STARTERS):
        return False
    return any(pat.match(line) for pat in HEADING_PATTERNS)


def is_new_paragraph(line_record, prev_line_record, gap_threshold=10):
    if prev_line_record is None:
        return True
    if line_record['page'] != prev_line_record['page']:
        prev_text = prev_line_record['text'].strip()
        return prev_text[-1:] in '.!?:;' if prev_text else True
    gap = line_record['gap_before']
    return gap is not None and gap > gap_threshold


def is_placeholder_text(text, alpha_ratio_threshold=0.2):
    letters = sum(1 for c in text if c.isalpha())
    return len(text) > 0 and (letters / len(text)) < alpha_ratio_threshold


def is_likely_fragment(text, min_absolute_len=20):
    text = text.strip()
    if not text:
        return True
    if text[0].islower():
        return True
    if len(text) < min_absolute_len:
        return True
    return False


def chunk_into_clauses(all_lines, min_chunk_chars=30):
    cleaned = []
    for rec in all_lines:
        t = clean_line(rec['text'])
        if t:
            cleaned.append({**rec, 'text': t})

    heading_idx = [i for i, r in enumerate(cleaned) if is_heading(r['text'])]
    chunks = []

    def join_lines(records):
        parts = []
        for r in records:
            t = r['text']
            if parts and parts[-1].endswith('-'):
                parts[-1] = parts[-1][:-1] + t
            else:
                parts.append(t)
        return ' '.join(parts)

    if heading_idx:
        for idx, start in enumerate(heading_idx):
            end = heading_idx[idx + 1] if idx + 1 < len(heading_idx) else len(cleaned)
            heading_text = cleaned[start]['text']
            body_text = join_lines(cleaned[start + 1:end])
            page_num = cleaned[start]['page']
            if len(body_text) >= min_chunk_chars:
                chunks.append({'chunk_id': len(chunks) + 1, 'section_heading': heading_text,
                                'clause_text': body_text, 'page_number': page_num,
                                'char_count': len(body_text)})
    else:
        current, prev, groups = [], None, []
        for r in cleaned:
            if is_new_paragraph(r, prev) and current:
                groups.append(current)
                current = []
            current.append(r)
            prev = r
        if current:
            groups.append(current)
        for i, group in enumerate(groups, start=1):
            text = join_lines(group)
            if len(text) >= min_chunk_chars:
                chunks.append({'chunk_id': i, 'section_heading': None, 'clause_text': text,
                                'page_number': group[0]['page'], 'char_count': len(text)})

    chunks = [c for c in chunks if not is_placeholder_text(c['clause_text'])]

    merged = []
    for c in chunks:
        if merged and is_likely_fragment(c['clause_text']):
            merged[-1]['clause_text'] = merged[-1]['clause_text'] + ' ' + c['clause_text']
            merged[-1]['char_count'] = len(merged[-1]['clause_text'])
        else:
            merged.append(dict(c))
    for i, c in enumerate(merged, start=1):
        c['chunk_id'] = i

    return pd.DataFrame(merged)


extracted_lines = extract_pages(pdf_path)
extracted_lines = strip_boilerplate(extracted_lines)
model1_output_df = chunk_into_clauses(extracted_lines)
print(f"MODEL 1 OUTPUT: {len(model1_output_df)} clause chunk(s) extracted")
print(model1_output_df[['chunk_id', 'section_heading', 'char_count']].to_string(index=False))

# ============================================================
# MODEL 2: CLAUSE CLASSIFICATION
# ============================================================

# Cell 5 - Load the Zero-Shot Classifier

Same model choice as the standalone Model 2 notebook (DeBERTa NLI, falling back to BART MNLI). This step downloads ~400MB on first run and is noticeably slow on CPU.

In [ ]:
import torch
from transformers import pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("WARNING: No GPU detected — this will be slow. Runtime > Change runtime type > T4 GPU.")

TARGET_CLAUSE_TYPES = [
    "compensation clause", "termination clause", "confidentiality clause",
    "non-compete clause", "intellectual property clause", "probation clause"
]

try:
    classifier = pipeline("zero-shot-classification", model="cross-encoder/nli-deberta-v3-small",
                           device=0 if device == "cuda" else -1)
    print("Classifier loaded (DeBERTa NLI zero-shot)")
except Exception as e:
    print(f"Primary model failed: {e}\nTrying fallback...")
    classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli",
                           device=0 if device == "cuda" else -1)
    print("Fallback classifier loaded (BART MNLI)")

# Cell 6 - Classify Model 1's Extracted Clauses

**Integration fix needed here:** the standalone Model 2 notebook's evaluation code assumed a `true_label` column (from CUAD ground-truth data used during testing). Real contracts coming out of Model 1 have no ground truth -- there's nothing to compare against. `classify_clauses()` itself never needed `true_label`, only some of the *display/eval* code around it did, so this is a one-line reminder not a bug -- but it would have thrown a `KeyError` the first time this ran on real pipeline output instead of a CUAD test set.

**Confidence threshold, added after a real MOM contract test:** the classifier has no way to say "none of the 6 types fit" -- it always forces a best guess, even for administrative content (commencement dates, signature blocks) that isn't really any of the 6 types. On that real run, all 7 wrong predictions had confidence <= 35%, and all 4 correct ones were >= 35%, so anything below that threshold is relabelled `unclassified` instead of a confident wrong answer.

In [ ]:
from tqdm import tqdm

CONFIDENCE_THRESHOLD = 0.35

def apply_confidence_threshold(df, threshold=CONFIDENCE_THRESHOLD):
    df = df.copy()
    below = df['confidence'] < threshold
    df.loc[below, 'predicted_label'] = 'unclassified'
    return df

def classify_clauses(df, classifier, clause_types, batch_size=8):
    predictions, confidences, all_scores = [], [], []
    clauses = df['clause_text'].tolist()
    print(f"Classifying {len(clauses)} clauses...")

    for i in tqdm(range(0, len(clauses), batch_size)):
        batch = clauses[i:i+batch_size]
        try:
            results = classifier(batch, clause_types, multi_label=False)
            if isinstance(results, dict):
                results = [results]
            for result in results:
                predictions.append(result['labels'][0])
                confidences.append(result['scores'][0])
                all_scores.append(dict(zip(result['labels'], result['scores'])))
        except Exception as e:
            print(f"Error on batch {i//batch_size}: {e}")
            for _ in batch:
                predictions.append("unknown")
                confidences.append(0.0)
                all_scores.append({})

    df = df.copy()
    df['predicted_label'] = predictions
    df['confidence'] = confidences
    for clause_type in clause_types:
        df[f'score_{clause_type.replace(" ", "_")}'] = [s.get(clause_type, 0.0) for s in all_scores]
    return df


model2_output_df = classify_clauses(model1_output_df, classifier, TARGET_CLAUSE_TYPES)

n_before = len(model2_output_df)
model2_output_df = apply_confidence_threshold(model2_output_df)
n_unclassified = (model2_output_df['predicted_label'] == 'unclassified').sum()
print(f"\n{n_unclassified} / {n_before} clauses fell below {CONFIDENCE_THRESHOLD:.0%} confidence "
      f"and were relabelled 'unclassified' instead of a forced guess")

print(f"\nMODEL 2 OUTPUT:")
# No true_label available for real contracts -- unlike the CUAD-based standalone test
print(model2_output_df[['clause_text', 'predicted_label', 'confidence']].to_string(index=False))

# Cell 7 - Sanity-Check Model 2's Output on Real Pipeline Data

In [ ]:
print("=" * 60)
print("MODEL 2 CONFIDENCE CHECK ON PIPELINE OUTPUT")
print("=" * 60)
print(f"Mean confidence:  {model2_output_df['confidence'].mean():.1%}")
print(f"Min confidence:   {model2_output_df['confidence'].min():.1%}")
low_conf = model2_output_df[model2_output_df['confidence'] < 0.4]
print(f"Clauses below 40% confidence: {len(low_conf)} / {len(model2_output_df)}")
if len(low_conf) > 0:
    print("\nThese are worth a manual look -- low confidence could mean either a genuinely")
    print("ambiguous clause, or extraction noise (a heading fragment, a merged/split clause)")
    print("feeding the classifier something odd:")
    for _, row in low_conf.iterrows():
        print(f"  - [{row['confidence']:.0%}] {row['clause_text'][:100]}...")
print("=" * 60)

# ============================================================
# MODEL 3: EXPLANATION GENERATION
# ============================================================

# Cell 8 - Set Up the LLM Client (Groq API)

Uses Groq's permanently free, rate-limited developer tier instead of a paid API so this pipeline can be run and evaluated by anyone without a paid account -- the same provider used in the deployed web application. Leaving the key blank still runs the pipeline end-to-end in clearly-labelled mock mode.

In [ ]:
client = None
MODEL_NAME = "openai/gpt-oss-120b"  # Groq deprecates/renames models fairly often --
                                     # check console.groq.com/docs/models if this 404s

try:
    api_key = getpass.getpass("Groq API key (leave blank to run in mock mode): ")
    if api_key.strip():
        import groq
        client = groq.Groq(api_key=api_key.strip())
        print("Client configured -- using live API calls.")
    else:
        print("No key entered -- running in MOCK MODE.")
except Exception as e:
    print(f"Falling back to mock mode ({e}).")

# Cell 9 - Generate Explanations for Model 2's Classified Clauses

Explanation and risk assessment only -- this cell never re-decides a clause's type, only explains the type Model 2 already assigned. A failed live call surfaces honestly as an ERROR state (see the function below), never a disguised guess.

In [ ]:
RISK_MAP = {
    "non-compete clause": {"level": "HIGH", "reason": "This clause restricts your ability to work for competitors after leaving. Check the duration and geographic scope carefully."},
    "intellectual property clause": {"level": "HIGH", "reason": "This may assign ownership of your personal projects or inventions to your employer. Check if it covers work done outside office hours."},
    "termination clause": {"level": "MEDIUM", "reason": "This defines how employment can be ended. Check notice periods and conditions for immediate termination without pay."},
    "confidentiality clause": {"level": "MEDIUM", "reason": "This restricts what you can discuss outside work. Check how broadly 'confidential information' is defined."},
    "compensation clause": {"level": "LOW", "reason": "This defines your pay and benefits. Verify the figures match what was discussed during your interview."},
    "probation clause": {"level": "LOW", "reason": "This sets the trial period terms. Check the duration and what happens at the end of probation."},
}

PROMPT_TEMPLATE = """You are helping a non-lawyer employee in Singapore understand one clause \
from their employment contract. You will be given the clause's type (as classified by an \
earlier model, which may occasionally be wrong) and its exact text.

Clause type: {clause_type}
Clause text: "{clause_text}"

Respond with ONLY a valid JSON object (no other text, no markdown fences) with exactly these keys:
- "risk_level": one of "LOW", "MEDIUM", "HIGH" -- based on the ACTUAL terms in this specific \
clause.
- "explanation": one or two plain-English sentences explaining what this specific clause means \
for the employee, referencing concrete details from the text where present.
- "watch_for": one short, concrete thing the employee should check or ask about, specific to \
what's actually written here.
"""


def _mock_llm_call(clause_text, predicted_label):
    """Clearly-labelled mock mode -- used only when no API key was ever provided,
    never as a silent disguise for a live failure (see generate_explanation below)."""
    base = RISK_MAP.get(predicted_label, {"level": "MEDIUM", "reason": "Review this clause carefully."})
    numbers = re.findall(r'\d+\s*(?:month|day|week|year)s?', clause_text, flags=re.I)
    detail = f" (note: this clause specifies {', '.join(numbers)})" if numbers else ""
    return json.dumps({"risk_level": base["level"], "explanation": base["reason"] + detail,
                        "watch_for": "Confirm this matches what was verbally agreed during your offer discussion."})


def generate_explanation(clause_text, predicted_label, client=None, model=MODEL_NAME, max_retries=2):
    if predicted_label == 'unclassified':
        return {
            "risk_level": "N/A",
            "explanation": "This clause didn't clearly match any of the 6 tracked categories "
                            "-- it may be administrative or procedural content.",
            "watch_for": "Skim this manually if it looks important; automatic risk assessment "
                         "wasn't confident enough to be reliable here."
        }

    # No key was ever provided -- clearly-labelled mock mode, not a failure.
    if client is None:
        raw = _mock_llm_call(clause_text, predicted_label)
        return json.loads(raw)

    # A real client exists -- any failure from here is a genuine error and must be
    # surfaced instead of silently disguised as mock output or a generic
    # type-based guess (an earlier version of this cell did exactly that, which is
    # the same category of issue caught and fixed in the deployed web app).
    prompt = PROMPT_TEMPLATE.format(clause_type=predicted_label, clause_text=clause_text[:1500])
    raw = None
    last_error = None
    for attempt in range(max_retries + 1):
        try:
            resp = client.chat.completions.create(
                model=model, max_tokens=300,
                messages=[{"role": "user", "content": prompt}]
            )
            raw = resp.choices[0].message.content
            break
        except Exception as e:
            last_error = e
            print(f"  [generate_explanation] API call failed (attempt {attempt + 1}/"
                  f"{max_retries + 1}): {type(e).__name__}: {e}")
            if attempt < max_retries:
                time.sleep(1)

    error_state = {
        "risk_level": "ERROR",
        "explanation": "This clause was classified but couldn't be explained automatically "
                        f"due to a connection issue "
                        f"({type(last_error).__name__ if last_error else 'unknown'}).",
        "watch_for": "Review this clause manually."
    }
    if raw is None:
        return error_state

    try:
        # Locate the JSON object by its first '{' and last '}' instead of assuming
        # the whole response is clean JSON -- necessary because reasoning-tuned models
        # sometimes prepend brief explanatory text before the structured answer.
        start, end = raw.find('{'), raw.rfind('}')
        if start == -1 or end == -1 or end < start:
            raise ValueError("No JSON object found in response")
        parsed = json.loads(raw[start:end + 1])
        assert parsed.get("risk_level") in ("LOW", "MEDIUM", "HIGH")
        assert parsed.get("explanation") and parsed.get("watch_for")
        return parsed
    except Exception as e:
        print(f"  [generate_explanation] Response wasn't valid JSON: {e}\nRaw: {raw[:200]}")
        return error_state


explanations = []
print(f"Generating explanations for {len(model2_output_df)} clauses "
      f"({'LIVE API' if client else 'MOCK MODE'})...\n")
for i, row in model2_output_df.iterrows():
    result = generate_explanation(row['clause_text'], row['predicted_label'], client=client)
    explanations.append(result)
    print(f"  [{i+1}/{len(model2_output_df)}] {row['predicted_label']} -> {result['risk_level']}")

final_df = model2_output_df.copy()
final_df['risk_level'] = [e['risk_level'] for e in explanations]
final_df['explanation'] = [e['explanation'] for e in explanations]
final_df['watch_for'] = [e['watch_for'] for e in explanations]
print("\nDone.")


# ============================================================
# FINAL OUTPUT
# ============================================================

# Cell 10 - Full Pipeline Sanity Check (end-to-end integrity)

Confirms nothing silently dropped rows between stages and that every clause made it through all three models with a complete record.

In [ ]:
print("=" * 60)
print("END-TO-END PIPELINE CHECK")
print("=" * 60)
print(f"Model 1 output (clauses extracted):     {len(model1_output_df)}")
print(f"Model 2 output (clauses classified):    {len(model2_output_df)}")
print(f"Model 3 output (clauses explained):     {len(final_df)}")

if len(model1_output_df) == len(model2_output_df) == len(final_df):
    print("\n[PASS] No clauses dropped across the pipeline")
else:
    print("\n[CHECK] Clause count changed between stages -- investigate where/why")

required_cols = ['clause_text', 'predicted_label', 'confidence', 'risk_level', 'explanation', 'watch_for']
missing = [c for c in required_cols if c not in final_df.columns]
print(f"\nRequired columns present: {'[PASS] all present' if not missing else f'[CHECK] missing {missing}'}")

incomplete = final_df[final_df[required_cols].isnull().any(axis=1)]
print(f"Rows with any missing field: {len(incomplete)} (should be 0)")
print("=" * 60)

# Cell 11 - Generate the Final User-Facing Report

In [ ]:
def generate_report(df):
    print("=" * 60)
    print("CONTRACT ANALYSIS AI — Employment Contract Review")
    print("=" * 60)
    print("DISCLAIMER: This tool is for informational purposes only")
    print("and does not constitute legal advice.")
    print("=" * 60)
    for i, row in df.iterrows():
        print(f"\nCLAUSE {i+1}: {row['predicted_label'].upper()}")
        print(f"  Risk Level:  {row['risk_level']}")
        print(f"  Text:        {row['clause_text'][:150]}{'...' if len(row['clause_text'])>150 else ''}")
        print(f"  Explanation: {row['explanation']}")
        print(f"  Watch for:   {row['watch_for']}")
        print(f"  Confidence:  {row['confidence']:.0%}")

generate_report(final_df)

# Cell 12 - Save Final Results

In [ ]:
output_path = "pipeline_final_output.csv"
final_df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")
try:
    files.download(output_path)
except Exception as e:
    print(f"(Download skipped — not running in Colab: {e})")

# Cell 13 - Summary of Results

In [ ]:
print("=" * 60)
print("END-TO-END PIPELINE SUMMARY")
print("=" * 60)
print(f"Source PDF:          {pdf_path}")
print(f"Clauses processed:   {len(final_df)}")

print("\n--- Integration issue found (historical) ---")
print("  Model 2's evaluation code assumed a 'true_label' column from CUAD ground-truth")
print("  data. Real pipeline output from Model 1 has no ground truth, so any display/eval")
print("  code referencing true_label would KeyError on real contracts. classify_clauses()")
print("  itself was unaffected -- only surrounding eval code needed the reminder.")

print("\n--- Current status (updated since the preliminary report) ---")
print("  - Model 1 now includes an OCR fallback (Tesseract LSTM) for scanned PDFs,")
print("    a genuine third pre-trained model alongside Model 2's classifier and")
print("    Model 3's LLM, and has been tested against two real SEC EDGAR employment")
print("    contracts in addition to the MOM template used earlier.")
print("  - Model 2 has since been fine-tuned as a supervised LEGAL-BERT classifier")
print("    (see Model2_LegalBERT_FineTuning.ipynb) and evaluated on real, held-out")
print("    CUAD data: a 33-68 percentage-point F1 gain over this zero-shot baseline")
print("    on every category with adequate test data (termination, confidentiality,")
print("    non-compete, IP). Compensation and probation could not be evaluated even")
print("    with the full real dataset -- CUAD's taxonomy doesn't meaningfully cover")
print("    those individual-employment concepts. The fine-tuned classifier is")
print("    validated but deliberately not deployed here (see the Design chapter for")
print("    why); this pipeline still runs the zero-shot classifier throughout.")
print("  - Model 3 now runs against Groq's live API (not just mock mode), with a")
print("    JSON-extraction fix and an honest ERROR state for genuine failures --")
print("    both added after live testing surfaced issues no offline test caught.")

print("\n--- What to check next ---")
print("  - This pipeline has now been run against real contracts beyond the")
print("    synthetic sample -- two known, reproducible zero-shot misclassifications")
print("    remain (a remuneration clause read as probation; an explicit non-compete")
print("    clause left unclassified), both already fixed by the fine-tuned classifier")
print("    validated above but not yet deployed into this pipeline.")
print("  - Usability testing with real, non-expert participants is a separate,")
print("    ongoing evaluation stream -- see the report's Evaluation chapter.")
print("=" * 60)
